# Step-by-step implementation

The following are the steps to implement Parent Document Retrieval (PDR):

1. **Prepare the data**
   1. Import necessary modules
   2. Set up the OpenAI API key
   3. Define the text embedding function
   4. Load text data
2. **Retrieve full documents**
   1. Full document splitting
   2. Vector store and storage setup
   3. Parent document retriever
   4. Adding documents
   5. Similarity search and retrieval
3. **Retrieve larger chunks**
   1. Parent document retriever
   2. Similarity search and retrieval
4. **Integrate with `RetrievalQA`**

## What is Parent Document Retrieval?

**Parent Document Retrieval (PDR)** is another way of decoupling *what you search over* from
*what you return* — this time along a chunk-size axis rather than a summary axis.

Small chunks embed precisely (a short, focused piece of text produces a sharp similarity
match), but they lack surrounding context when handed to the LLM. Large chunks carry more
context but embed poorly (a long chunk blends many topics into one vague vector, so
similarity search is less accurate). PDR gets both:

- **`child_splitter`** — splits documents into *small* chunks that get embedded into the
  **vectorstore** (`Chroma`) for precise similarity search.
- **`parent_splitter`** (optional) — splits documents into *larger* parent chunks (or, if
  omitted, the whole original document is the "parent"). These live in the **docstore**
  (`InMemoryStore`), keyed by a `parent_id` that each child chunk carries in its metadata.
- **`ParentDocumentRetriever`** — searches the vectorstore over child chunks, then uses each
  match's `parent_id` to fetch and return the larger parent chunk (or full document) from the
  docstore — so the LLM sees full context, not an isolated fragment.

![Parent Document Retrieval diagram](img/parent_document_retrieval.png)

**Flow in this notebook:**
1. Load source documents (`shared_data/*.txt`).
2. **Full-document retrieval** — split only with a `child_splitter`; each child chunk's
   parent is the *entire original document*.
3. **Larger-chunk retrieval** — split with both a `parent_splitter` (e.g. 2000 chars) and a
   `child_splitter` (e.g. 400 chars), so parents are large chunks rather than whole documents.
4. Query either retriever: it searches the small child-chunk embeddings, then returns the
   corresponding larger parent chunk/document.
5. Wire the retriever into a `RetrievalQA` chain to answer questions end-to-end.

## Prepare the data

### i) Import necessary modules

In [1]:
from langchain_core.documents import Document
from langchain_chroma import Chroma
from langchain_classic.retrievers import ParentDocumentRetriever
from langchain_classic.chains import RetrievalQA
from langchain_openai import OpenAI
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.stores import InMemoryStore
from langchain_community.document_loaders import TextLoader
from langchain_openai import OpenAIEmbeddings
import os

C:\Users\soura\AppData\Local\Temp\ipykernel_26836\617324231.py:8: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


### ii) Set up the OpenAI API key

In [2]:
from dotenv import load_dotenv
load_dotenv(override=True)

OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY", "")
if OPENAI_API_KEY == "":
    raise ValueError("Please set the OPENAI_API_KEY environment variable")

### iii) Define the text embedding function

In [3]:
embeddings = OpenAIEmbeddings()

### vi) Load text data

In [4]:
loaders = [
    TextLoader('../shared_data/blog.langchain.dev_announcing-langsmith_.txt', encoding="utf-8"),
    TextLoader('../shared_data/blog.langchain.dev_automating-web-research_.txt', encoding="utf-8"),
]

docs = []
for l in loaders:
    docs.extend(l.load())

## 2. Retrieve full documents

### i) Full document splitting

In [5]:
child_splitter = RecursiveCharacterTextSplitter(chunk_size=400)

### ii) Vector store and storage setup

In [6]:
vectorstore = Chroma(
    collection_name="full_documents",
    embedding_function=OpenAIEmbeddings()
)

store = InMemoryStore()

### iii) Parent document retriever

In [7]:
full_doc_retriever = ParentDocumentRetriever(
    vectorstore=vectorstore,
    docstore=store,
    child_splitter=child_splitter
)

### iv) Adding documents

In [8]:
full_doc_retriever.add_documents(docs)

print(list(store.yield_keys()))  # List document IDs in the store

['44cafc98-4c1e-45a5-a836-71e29dd92fe4', 'c315821a-afec-410e-a0b6-b021fc8b93a1']


### v) Similarity search and retrieval

In [9]:
sub_docs = vectorstore.similarity_search("What is LangSmith?", k=2)
print(len(sub_docs))

print(sub_docs[0].page_content)

retrieved_docs = full_doc_retriever.invoke("What is LangSmith?")

print(len(retrieved_docs[0].page_content))
print(retrieved_docs[0].page_content)

2
Today, we’re introducing LangSmith, a platform to help developers close the gap between prototype and production. It’s designed for building and iterating on products that can harness the power–and wrangle the complexity–of LLMs.
11652
URL: https://blog.langchain.dev/announcing-langsmith/
Title: Announcing LangSmith, a unified platform for debugging, testing, evaluating, and monitoring your LLM applications

LangChain exists to make it as easy as possible to develop LLM-powered applications.

We started with an open-source Python package when the main blocker for building LLM-powered applications was getting a simple prototype working. We remember seeing Nat Friedman tweet in late 2022 that there was “not enough tinkering happening.” The LangChain open-source packages are aimed at addressing this and we see lots of tinkering happening now (Nat agrees)–people are building everything from chatbots over internal company documents to an AI dungeon master for a Dungeons and Dragons game.


## 3. Retrieve larger chunks

### i) Parent document retriever

In [10]:
parent_splitter = RecursiveCharacterTextSplitter(chunk_size=2000)
child_splitter = RecursiveCharacterTextSplitter(chunk_size=400)

vectorstore = Chroma(
    collection_name="split_parents",
    embedding_function=OpenAIEmbeddings()
)

store = InMemoryStore()

big_chunks_retriever = ParentDocumentRetriever(
    vectorstore=vectorstore,
    docstore=store,
    child_splitter=child_splitter,
    parent_splitter=parent_splitter
)

# Adding documents
big_chunks_retriever.add_documents(docs)
print(len(list(store.yield_keys())))  # List document IDs in the store

10


### ii) Similarity search and retrieval

In [11]:
sub_docs = vectorstore.similarity_search("What is LangSmith?", k=2)
print(len(sub_docs))

print(sub_docs[0].page_content)

retrieved_docs = big_chunks_retriever.invoke("What is LangSmith?")
print(len(retrieved_docs))

print(len(retrieved_docs[0].page_content))
print(retrieved_docs[0].page_content)

2
Today, we’re introducing LangSmith, a platform to help developers close the gap between prototype and production. It’s designed for building and iterating on products that can harness the power–and wrangle the complexity–of LLMs.
3
1869
URL: https://blog.langchain.dev/announcing-langsmith/
Title: Announcing LangSmith, a unified platform for debugging, testing, evaluating, and monitoring your LLM applications

LangChain exists to make it as easy as possible to develop LLM-powered applications.

We started with an open-source Python package when the main blocker for building LLM-powered applications was getting a simple prototype working. We remember seeing Nat Friedman tweet in late 2022 that there was “not enough tinkering happening.” The LangChain open-source packages are aimed at addressing this and we see lots of tinkering happening now (Nat agrees)–people are building everything from chatbots over internal company documents to an AI dungeon master for a Dungeons and Dragons game.

## 4. Integrate with `RetrievalQA`

In [12]:
qa = RetrievalQA.from_chain_type(llm=OpenAI(),
                                chain_type="stuff",
                                retriever=big_chunks_retriever)

query = "What is LangSmith?"

response = qa.invoke(query)
print(response)

{'query': 'What is LangSmith?', 'result': ' LangSmith is a platform designed to help developers debug, test, evaluate, and monitor their LLM-powered applications. It provides full visibility into model performance, including inputs and outputs, latency, and token usage. It also offers features like debugging and tracking user interactions to help teams identify and resolve issues with their applications.'}
